# **Car Price Prediction using Machine Learning**
# Objective
The objective of this project is to build a machine learning regression model that estimates the selling price of cars (Price_USD) based on their technical specifications and market characteristics such as brand, fuel type, engine power, and mileage.

---

# Car Price Prediction Project
#Problem Statement
The goal involves predicting a continuous numerical value (Car Price) based on various features like horsepower, engine displacement, transmission type, and manufacturing year.

# Target Variable

The target variable is Price_USD, which represents the market selling price of the car in US Dollars.

# Data Source
Dataset obtained from Kaggle (Car Price Classification Ready Dataset)

# Interpretation:



*   Problem Type: This is a Supervised Learning Regression problem because we are predicting a continuous quantity (Price) using labeled historical data.


*   Why this target is important? Accurately predicting the price allows buyers to determine fair market value and sellers (or dealerships) to set competitive prices.



# **Importing all the necessary libraries**

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

These libraries are used for data manipulation, visualization, preprocessing, model building, and evaluation. Scikit-learn provides a consistent and reproducible pipeline for machine learning experiments.

---

# **Setup & Dataset Loading**

In [ ]:
df = pd.read_csv("/content/global_cars_enhanced.csv")

# View first rows
print(df.head())

# Check dataset info
print(df.info())


In [ ]:
print(df.isnull().sum())


In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(exclude=np.number).columns

# Fill numerical missing values with mean
num_imputer = SimpleImputer(strategy="mean")
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Fill categorical missing values with most frequent value
cat_imputer = SimpleImputer(strategy="most_frequent")
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])


In [ ]:
df = df.drop_duplicates()

In [ ]:
target = "Price_USD"
X = df.drop(columns=[target])
y = df[target]

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
df.size

In [ ]:
df.columns

In [ ]:
df['Car_ID'].isnull().sum()

In [ ]:
print(df.describe())

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

# **Success Metrics**
The project imports specific metrics to evaluate model performance:


*   List itemMean Absolute Error (MAE): Calculates the average absolute difference between predicted and actual prices.



*   Mean Squared Error (MSE): Penalizes larger errors more heavily than smaller ones.


*   R² Score (Coefficient of Determination): Measures how well the independent variables explain the variance in the car price.





Interpretation:


*   Why R² is important: It gives a quick summary (0 to 1) of how "good" the model is fitting the data compared to a simple average baseline.


*   Why MAE/MSE? In car pricing, we want to minimize the dollar deviation from the actual market price.

---



# **Data Understanding**

In [ ]:
df.shape

In [ ]:
df.columns

The dataset contains columns describing the car's physical and performance attributes.

---

# Data Dictionary


| Column Name               | Type                 | Description                                                |
| ------------------------- | -------------------- | ---------------------------------------------------------- |
| **Car_ID**                | Integer              | Unique identifier for each car record                      |
| **Brand**                 | Categorical (String) | Car manufacturer name (Toyota, BMW, etc.)                  |
| **Manufacture_Year**      | Integer              | Year the car was manufactured                              |
| **Body_Type**             | Categorical          | Type of car body (SUV, Sedan, Hatchback, etc.)             |
| **Fuel_Type**             | Categorical          | Fuel used (Petrol, Diesel, Electric, Hybrid)               |
| **Transmission**          | Categorical          | Gear type (Manual or Automatic)                            |
| **Engine_CC**             | Numeric              | Engine capacity in cubic centimeters                       |
| **Horsepower**            | Numeric              | Engine power output in horsepower                          |
| **Mileage_km_per_l**      | Numeric              | Fuel efficiency (km per liter)                             |
| **Price_USD**             | Numeric              | Car price in USD *(Target Variable)*                       |
| **Manufacturing_Country** | Categorical          | Country where the car is made                              |
| **Car_Age**               | Numeric              | Age of the car (Current Year − Manufacture Year)           |
| **Price_Category**        | Categorical          | Price segment (Budget, Mid-Range, Premium)                 |
| **HP_per_CC**             | Numeric              | Ratio of horsepower to engine size (performance indicator) |
| **Age_Category**          | Categorical          | Age group of car (New, Mid, Old)                           |
| **Efficiency_Score**      | Numeric              | Combined efficiency metric based on mileage & performance  |


In [ ]:
df.dtypes

The dataset contains a mix of numerical (Engine_CC, Horsepower) and categorical (Brand, Fuel_Type) features. Understanding these types is crucial because they require different preprocessing steps (scaling for numbers, encoding for categories).

In [ ]:
numerical_features = df.select_dtypes(include=['int64','float64']).columns
categorical_features = df.select_dtypes(include=['object']).columns

print("Numerical Features:\n", numerical_features)
print("\nCategorical Features:\n", categorical_features)

In [ ]:
categorical_features = df.select_dtypes(include='object').columns

for col in categorical_features:
    print(f"\nValue counts for {col}:\n")
    print(df[col].value_counts())

# **Data Quality Assessment**

In [ ]:
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0].sort_values(ascending=False))

In [ ]:
(missing_values / len(df)) * 100

The code checks for null values. Missing data can bias the model, so it must be handled (imputed) rather than ignored.

---

In [ ]:
df.duplicated().sum()

Duplicate rows are removed to prevent the model from memorizing repeated data points, which would lead to overfitting (performing well on training data but failing on new data).

---

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.boxplot(x=df['Price_USD'])

plt.title("Outlier Detection in Target Variable (Price_USD)")
plt.show()

The target variable (Price_USD) shows the presence of extreme values and right-skewness, which is expected in the car market due to luxury and high-performance vehicles.

These outliers represent valid high-end cars rather than data errors, but they may influence the model to be biased toward higher prices if not handled correctly.

---

#**Data Preprocessing & Cleaning**

In [ ]:
df = df.drop(columns=[
    'Car_ID',          # just an ID, no prediction value
    'Price_Category',  # derived from Price_USD (target leakage)
    'Age_Category'     # derived from Car_Age
], errors='ignore')

Text-heavy columns and pricing-derived features were removed as they do not directly contribute to numerical modeling and may introduce data leakage.

Removing such features helps the model learn genuine relationships rather than memorizing pricing information.

In [ ]:
df['Price_USD'] = pd.to_numeric(df['Price_USD'], errors='coerce')

This command forces the Price_USD column into a numeric format, converting any invalid text or non-parseable values into NaN (missing values) to ensure the target variable is clean for mathematical modeling.

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

Missing numerical values were imputed using the median to reduce sensitivity to outliers.

Missing categorical values were imputed using the mode to preserve the most frequent category.

---

# **Exploratory Data Analysis (EDA)**

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['Price_USD'], kde=True)

plt.title("Distribution of Car Prices")
plt.xlabel("Price (USD)")
plt.ylabel("Frequency")
plt.show()

The distribution of car prices is heavily right-skewed (positively skewed).

The majority of cars are concentrated in the lower price range, while a long tail extends to the right, representing expensive vehicles. This suggests that a log transformation might be beneficial to normalize the distribution for regression modeling.

---

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x='Brand', y='Price_USD', data=df)

plt.xticks(rotation=45)
plt.title("Price Distribution by Brand")
plt.show()

There is a significant variation in price ranges across different brands.

Luxury brands (like BMW, Audi, Mercedes) show a higher median price and a wider spread, indicating they offer both mid-range and premium models. Economy brands have lower median prices and tighter distributions. This confirms that Brand is a strong predictor of price.

---

In [ ]:
numerical_cols = [
    'Manufacture_Year',
    'Engine_CC',
    'Horsepower',
    'Mileage_km_per_l',
    'Price_USD',
    'Car_Age',
    'HP_per_CC',
    'Efficiency_Score'
]

df[numerical_cols].hist(figsize=(12,8), bins=30)

plt.tight_layout()
plt.show()

Most numerical features such as Engine_CC and Horsepower show right-skewed distributions, indicating that most cars have standard engine sizes, while a few have very large, powerful engines.

Manufacture_Year is skewed towards recent years, suggesting the dataset consists mostly of newer cars. Mileage_km_per_l appears more normally distributed.

---

In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x='Horsepower', y='Price_USD', data=df)

plt.title("Horsepower vs Price")
plt.show()

There is a clear positive relationship between horsepower and car price, indicating that higher performance generally commands a higher market value.

The relationship appears somewhat linear but shows increasing variance at higher horsepower levels (heteroscedasticity), meaning price prediction becomes more variable for extremely powerful cars.

---

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(df.corr(numeric_only=True),
            annot=True,
            cmap='coolwarm',
            fmt=".2f")

plt.title("Correlation Heatmap")
plt.show()

The correlation heatmap shows that Horsepower and Engine_CC have the strongest positive correlation with Price_USD, making them key features for prediction.

Mileage_km_per_l shows a negative correlation with price, as more expensive/powerful cars tend to be less fuel-efficient. High correlation between Horsepower and Engine_CC suggests multicollinearity, which ensemble models like Random Forest can handle effectively.

---

In [ ]:
df.isnull().sum()

In [ ]:
numeric_features = [
    'Manufacture_Year',
    'Engine_CC',
    'Horsepower',
    'Mileage_km_per_l',
    'Car_Age',
    'HP_per_CC',
    'Efficiency_Score'
]

categorical_features = [
    'Brand',
    'Body_Type',
    'Fuel_Type',
    'Transmission',
    'Manufacturing_Country'
]

In [ ]:
from sklearn.compose import ColumnTransformer
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

---

#**Feature Engineering**

In [ ]:
df['Car_Age'] = 2025 - df['Manufacture_Year']

Converted Manufacture_Year into Age because the "age" of a car is more directly related to depreciation than the specific year it was made.

In [ ]:
df['HP_per_CC'] = df['Horsepower'] / df['Engine_CC']

A derived metric representing engine efficiency (power per unit of size).

In [ ]:
df['Efficiency_Score'] = df['Mileage_km_per_l'] * df['HP_per_CC']

In [ ]:
luxury_brands = ['BMW','Mercedes','Audi']

df['Is_Luxury'] = df['Brand'].apply(
    lambda x: 1 if x in luxury_brands else 0
)

A custom binary flag created to explicitly tell the model if a brand is considered "Luxury". This helps the model distinguish why a BMW might cost more than a Toyota with similar specs.

In [ ]:
df['Age_Group'] = pd.cut(
    df['Car_Age'],
    bins=[0,5,10,20],
    labels=['New','Mid','Old']
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

In [ ]:
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

---

#**Model Selection**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor()
}

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

# --- Corrected Data Preparation and Model Training ---

# 1. Define X and y from the current 'df' after all feature engineering and column drops.
#    Ensure 'df' is still a DataFrame at this point.
X = df.drop(columns=['Price_USD'])
y = df['Price_USD']

# Drop 'Age_Group' as a diagnostic step due to negative R2 scores and its NaN values
# This temporary removal will help determine if the feature or its handling is problematic.
X = X.drop(columns=['Age_Group'], errors='ignore')

# Drop 'Manufacture_Year' as it's highly correlated with 'Car_Age'
# Keeping both can lead to multicollinearity, especially for linear models.
X = X.drop(columns=['Manufacture_Year'], errors='ignore')

# --- Debugging Step: Check for NaN/Inf values in X before preprocessing ---
# Check for NaN values
print("NaN values in X before preprocessing:")
print(X.isnull().sum()[X.isnull().sum() > 0])

# Check for Inf values
print("\nInf values in X before preprocessing:")
print(X.isin([np.inf, -np.inf]).sum()[X.isin([np.inf, -np.inf]).sum() > 0])
# --- End Debugging Step ---

# 2. Dynamically update numeric_features and categorical_features based on the current X DataFrame.
#    This accounts for engineered features like 'Is_Luxury' etc. and the removal of 'Age_Group'.
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 3. Modify numeric_transformer to include StandardScaler within the pipeline.
#    This ensures scaling happens *after* ColumnTransformer selects the columns.
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()) # StandardScaler is now part of the pipeline
])

# 4. Define categorical_transformer (already defined previously, but ensuring consistency).
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 5. Redefine preprocessor with the updated transformers and features.
#    'remainder=\'passthrough\'' ensures any columns not explicitly listed are kept (if any),
#    though with dynamic selection, it might not be strictly necessary.
preprocessor = ColumnTransformer(
    [
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

# 6. Perform train_test_split with the correct X and y (as DataFrames).
#    This ensures X_train and X_test are DataFrames when fed to the preprocessor.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Define models dictionary (from cell QqUfi8MWt-9dj)
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42), # Added random_state for reproducibility
    "Gradient Boosting": GradientBoostingRegressor(random_state=42) # Added random_state
}

# 8. Loop through models and evaluate
print("\nModel Evaluation Results:")
for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    print(f"{name} R2 Score: {r2_score(y_test, preds):.4f}")

The project selects three distinct models to compare:

**Linear Regression:**

* **Why?** Serves as a baseline. It assumes a straight-line relationship between features and price.

* **Pros/Cons:** Simple and interpretable, but often underfits complex real-world data like car prices.

**Random Forest Regressor:**

* **Why?** An ensemble method that averages multiple decision trees.

* **Pros/Cons:** Handles non-linear relationships well, robust to outliers, and reduces the risk of overfitting compared to a single decision tree.

**Gradient Boosting / Decision Tree:**

* **Why?** Used to compare against the Random Forest to find the best balance of bias and variance.

---

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline

dt_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42))
])

# Train the model
dt_model.fit(X_train, y_train)

The Decision Tree model is trained to capture non-linear patterns by splitting data based on rules (e.g., "If Horsepower > 300, Price is High").

* **Strength:** It can model complex interactions between features (e.g., how the value of a 'Luxury' brand changes with 'Age').

* **Weakness:** A single tree has high variance, meaning it tends to "memorize" the specific cars in the training set (overfitting), making it less reliable for new data compared to an ensemble.

---

# **Model Training**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

base_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

base_model.fit(X_train, y_train)

The Linear Regression model is trained as a baseline to establish a reference performance level.

* **Role:** It assumes a straightforward linear relationship between features (like Car_Age or Horsepower) and the target Price_USD.

* **Why use it?** If this simple model performs well, it suggests the pricing logic is simple. If it performs poorly (which is expected here), it confirms the need for complex, non-linear models.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

adv_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

adv_model.fit(X_train, y_train)

The Random Forest Regressor is trained as the final advanced model.

* **Mechanism:** It builds multiple decision trees (200 in this case) and averages their predictions.

* **Advantage:** This Ensemble Learning approach drastically reduces the risk of overfitting (high variance) seen in single decision trees while maintaining the ability to model complex, non-linear market behaviors.

* **Stability:** The fixed random_state=42 ensures that the model produces consistent results every time it is trained, which is critical for validation.

In [ ]:
from sklearn.metrics import r2_score

base_preds = base_model.predict(X_test)
adv_preds = adv_model.predict(X_test)

print("Base Model R2:", r2_score(y_test, base_preds))
print("Advanced Model R2:", r2_score(y_test, adv_preds))

---

# **Model Evaluation**

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(model, X_test, y_test):

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    return mae, rmse, r2

**Interpretation of Metrics:**

* **MAE (Mean Absolute Error):** The average dollar amount the model is off by. Lower is better.

* **RMSE (Root Mean Squared Error):** Penalizes large errors (e.g., mispricing a luxury car by a huge margin).

* **R² Score:** Represents accuracy (0 to 1). A score of 0.90 would mean the model explains 90% of the price variation, which is excellent.

In [ ]:
print("Linear Regression Performance")
lr_mae, lr_rmse, lr_r2 = evaluate_model(
    base_model, X_test, y_test
)

print("\nRandom Forest Performance")
rf_mae, rf_rmse, rf_r2 = evaluate_model(
    adv_model, X_test, y_test
)

print("\nDecision Tree Performance")
dt_mae, dt_rmse, dt_r2 = evaluate_model(
    dt_model, X_test, y_test
)

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Decision Tree'],
    'MAE': [lr_mae, rf_mae, dt_mae],
    'RMSE': [lr_rmse, rf_rmse, dt_rmse],
    'R2 Score': [lr_r2, rf_r2, dt_r2]
})

print("\nModel Comparison:")
print(comparison_df)



**Comparison Results:**

The **Linear Regression model** serves as a baseline, indicating how well a simple linear relationship explains car prices. Its lower R² score confirms that car pricing is too complex for a straight line.

The **Random Forest Regressor** achieves the lowest errors (MAE/RMSE) and the highest R² score. This confirms it is the superior model because it successfully captures non-linear relationships (like the exponential price increase of supercars) and handles outliers better than the single Decision Tree.

**Conclusion:** We select Random Forest as the final model for deployment due to its balance of accuracy and stability.

---

# **Model Interpretation & Explainability**

In [ ]:
import pandas as pd

# Extract trained Random Forest model
rf_model = adv_model.named_steps['model']

# Get feature names after preprocessing
feature_names = adv_model.named_steps[
    'preprocessor'
].get_feature_names_out()

# Get importances
importances = rf_model.feature_importances_

# Create dataframe
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df.head(10))

This step reveals which factors drive car prices the most.

* **Top Features:** Likely Horsepower, Car_Age, and Engine_CC.

* **Insight:** This confirms that performance and age are the primary deciders of value in the used car market, aligning with real-world logic.

In [ ]:
import matplotlib.pyplot as plt

# Predictions from best model (Random Forest)
y_pred = adv_model.predict(X_test)

plt.figure(figsize=(6,6))

plt.scatter(y_test, y_pred, alpha=0.4)

# Perfect prediction line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='red', linestyle='--'
)

plt.xlabel("Actual Car Price (USD)")
plt.ylabel("Predicted Car Price (USD)")
plt.title("Actual vs Predicted Car Prices")

plt.show()

The scatter plot compares actual prices against predicted prices.

The points cluster tightly around the diagonal red line (perfect prediction), indicating the model has high accuracy. Deviations from the line represent prediction errors, with some spread at the higher end suggesting the model might slightly struggle with accurately pricing very expensive luxury cars.

---

In [ ]:
# Target
y = df["Price_USD"]

# Features
X = df[[
    "Manufacture_Year",
    "Engine_CC",
    "Horsepower",
    "Mileage_km_per_l",
    "Car_Age",
    "HP_per_CC",
    "Efficiency_Score"
]]

from sklearn.ensemble import RandomForestRegressor

# Model
model = RandomForestRegressor(
    n_estimators=40,
    max_depth=10,
    random_state=42
)

# Train
model.fit(X, y)

import joblib

joblib.dump(adv_model, "car_price_pipeline.pkl")

The trained Random Forest pipeline is serialized and saved as a .pkl file. This allows the model to be transported and used in a real-world application without needing to retrain it every time.

In [ ]:
!pip install streamlit

In [ ]:
!npm install -g localtunnel

In [ ]:
%%writefile app.py

import streamlit as st
import joblib
import pandas as pd

model = joblib.load("car_price_pipeline.pkl")

st.title("🚗 Car Price Prediction App")

brand = st.selectbox("Brand", ["Toyota","BMW","Audi","Honda"])
body = st.selectbox("Body Type", ["SUV","Sedan","Hatchback"])
fuel = st.selectbox("Fuel Type", ["Petrol","Diesel","Electric"])
trans = st.selectbox("Transmission", ["Manual","Automatic"])
country = st.selectbox("Country", ["USA","Germany","Japan"])

year = st.number_input("Manufacture Year", 2000, 2025)
engine = st.number_input("Engine CC", 800, 5000)
hp = st.number_input("Horsepower", 50, 1000)
mileage = st.number_input("Mileage", 5.0, 40.0)

if st.button("Predict Price"):

    car_age = 2025 - year
    hp_per_cc = hp / engine
    eff = mileage * hp_per_cc

    # ⭐ Luxury flag (THIS WAS MISSING)
    is_luxury = 1 if brand in ["BMW","Audi","Mercedes"] else 0

    data = pd.DataFrame([{
        "Brand": brand,
        "Body_Type": body,
        "Fuel_Type": fuel,
        "Transmission": trans,
        "Manufacturing_Country": country,

        "Engine_CC": engine,
        "Horsepower": hp,
        "Mileage_km_per_l": mileage,

        "Car_Age": car_age,
        "HP_per_CC": hp_per_cc,
        "Efficiency_Score": eff,
        "Is_Luxury": is_luxury
    }])

    # ensure column order matches training
    data = data[model.feature_names_in_]

    pred = model.predict(data)[0]

    st.success(f"💰 Estimated Price: ${pred:,.2f}")

* The model is saved and loaded into a user-friendly web interface.

* **Real-world Application:** This allows a non-technical user (like a car buyer) to enter details (Year, Brand, Mileage) and get an instant estimated fair price, converting the complex code into a practical tool.

In [ ]:
import sklearn
import joblib
print(sklearn.__version__)